##### ======================= STAGE 03 - DATA CLEANING =================================== ######

This stage will cover:

- Removing duplicates

- Replacing negative kWh with NULL

- Filling missing regions as “Unknown”

- Capping extreme bills at 99.5th percentile

- Normalising timestamps

In [2]:
import pandas as pd
from sqlalchemy import text
import yaml

import sys
import os
sys.path.append(os.path.abspath(".."))

from src.mysql_db_utils import get_mysql_engine

# ESTABLISH CONNECTION TO MYSQL DATABASE
engine = get_mysql_engine()

# READ THE ENTIRE DATA FROM MYSQL DATABASE TO THE NOTEBOOK
query = text("SELECT * FROM energy_consumption_data;")
energy_df = pd.read_sql(query, engine)
energy_df.head()

,customer_id,region,meter_id,timestamp_utc,kwh,tariff_plan,is_smart_meter,outage_minutes_last_24h,bill_amount_eur,complaint_flag
0,201,East,MID201,2026-01-01 00:00:00,2.831160,fixed,1,15,0.509609,0
1,201,East,MID201,2026-01-01 01:00:00,1.939790,fixed,1,10,0.349162,0
2,201,East,MID201,2026-01-01 02:00:00,1.823063,fixed,1,11,0.328151,0
3,201,East,MID201,2026-01-01 03:00:00,0.629419,fixed,1,7,0.113295,0
4,201,NaN,MID201,2026-01-01 04:00:00,1.423434,fixed,1,10,0.256218,0


In [3]:
df = energy_df.copy()

In [ ]:
# CHECK FOR DUPLICATES
df.duplicated().sum()

np.int64(1430)

In [14]:
# CHECKING FOR NEGATIVE KWH READINGS
df[df['kwh'] < 0]
len(df[df['kwh'] < 0])

print(len(df[df['kwh'] < 0])/len(df)*100)

0.49436193619361934


###### *The assumptions is that, negative `kwh` readings are invalid and should be treated as missing then imputed od dropped depeneding on the percentage.

In [15]:
# CHECKING FOR MISSING VALUES
df.isnull().sum()

customer_id                   0
region                     2948
meter_id                      0
timestamp_utc                 0
kwh                           0
tariff_plan                   0
is_smart_meter                0
outage_minutes_last_24h       0
bill_amount_eur               0
complaint_flag                0
dtype: int64

In [18]:
# CHECK FOR EXTREME OUTLIERS
print(df.min())
print("---------------------------------")
print(df.max())
print("Assumption is thta, any value above the 99.5th percentile is an extreme outlier and it will be capped to the 99.5th percentile value.")

customer_id                                201
region                                    East
meter_id                                MID201
timestamp_utc              2026-01-01 00:00:00
kwh                                   -6.56184
tariff_plan                              fixed
is_smart_meter                               0
outage_minutes_last_24h                      0
bill_amount_eur                       0.000584
complaint_flag                               0
dtype: object
---------------------------------
customer_id                                300
region                                    West
meter_id                                MID300
timestamp_utc              2026-03-01 23:00:00
kwh                                  13.814755
tariff_plan                           variable
is_smart_meter                               1
outage_minutes_last_24h                     32
bill_amount_eur                      10.042691
complaint_flag                               1
dtype: objec

In [19]:
# DATA CLEANING FUNCTION
def data_cleaning_pipeline(df: pd.DataFrame) -> pd.DataFrame:
    # REMOVE DUPLICATES
    df = df.drop_duplicates().copy()
    # HANDLE NEGATIVE KWH READINGS
    df.loc[df['kwh'] < 0, 'kwh'] = pd.NA
    df['kwh'] = df['kwh'].fillna(df['kwh'].median())
    # HANDLE MISSING VALUES IN THE REGION COLUMN
    df['region'] = df['region'].fillna(df['region'].mode()[0])

    # HANDLE EXTREME OUTLIERS IN THE BILL AMOUNT COLUMN
    upper_cap_bill = df['bill_amount_eur'].quantile(0.995)
    df.loc[df['bill_amount_eur'] > upper_cap_bill, 'bill_amount_eur'] = upper_cap_bill

    return df

In [ ]:
df = data_cleaning_pipeline(df)
df.isnull().sum()

(144010, 10)

In [22]:
# SAVE CLEANED DATA TO CSV

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
data_path = os.path.join(BASE_DIR, "data", "cleaned", "energy_cleaned_data.csv")

os.makedirs(os.path.dirname(data_path), exist_ok=True)

df.to_csv(data_path, index=False)